Pasar de un Notebook a una aplicación web real con Streamlit es el paso definitivo para convertir un código de Python en una herramienta que cualquier recepcionista o gerente de hotel podría usar. Pero antes de ponernos con ello, vamos a realizar una prueba de que el modelo funciona.

In [1]:
import pickle
import pandas as pd
import numpy as np

In [ ]:
# 1. CARGAR
with open('modelo_final_hotel.pkl', 'rb') as f:
    modelo = pickle.load(f)
with open('escalador_hotel.pkl', 'rb') as f:
    scaler = pickle.load(f)
with open('columnas_modelo_final_hotel.pkl', 'rb') as f:
    columnas = pickle.load(f)

# 2. SIMULACIÓN DE RESERVA SEGÚN TU GRÁFICO REAL
# Inicializamos todas en 0 respetando la estructura del modelo
reserva_dic = {col: [0] for col in columnas}

# --- VARIABLES DE MAYOR IMPACTO (SEGÚN TU GRÁFICO) ---
reserva_dic['room_changed'] = [1]                 # 1. LA MÁS IMPORTANTE (Tu Feature Engineering)
reserva_dic['required_car_parking_spaces'] = [0] # 2. Muy relevante: 0 significa más riesgo
reserva_dic['country'] = [2]                     # 3. 2= PRT  
reserva_dic['customer_type_Transient'] = [1]     # 4. Clientes de paso (más volátiles)
reserva_dic['deposit_type_Non Refund'] = [0]     # 5. Sin depósito = más riesgo
reserva_dic['market_segment'] = [2]              # 6. Canal de venta
reserva_dic['total_of_special_requests'] = [0]   # 7. Pocas peticiones = menos compromiso
reserva_dic['agent'] = [9]                       # 8. ID del agente
reserva_dic['lead_time'] = [150]                 # 9. Antelación
reserva_dic['had_prev_cancellations'] = [0]      # 10. Historial previo

# 3. PROCESAMIENTO TÉCNICO
df_test = pd.DataFrame(reserva_dic)
df_test = df_test[columnas] # Orden exacto de las columnas

# Escalado
df_test_scaled = scaler.transform(df_test)

# Predicción
pred = modelo.predict(df_test_scaled)
prob = modelo.predict_proba(df_test_scaled)

# 4. DIAGNÓSTICO
print("=== DIAGNÓSTICO DEL MODELO (XGBOOST) ===")
print(f"Variable estrella (room_changed): {reserva_dic['room_changed'][0]}")
print(f"Resultado: {'CANCELARÁ' if pred[0] == 1 else 'CHECK-IN'}")
print(f"Probabilidad de Cancelación: {prob[0][1]*100:.2f}%")

=== DIAGNÓSTICO DEL MODELO (ORDEN CORREGIDO) ===
Variable estrella (room_changed): 1
Resultado: CANCELARÁ
Probabilidad de Cancelación: 81.78%
